In [1]:

!pip install -q streamlit pandas plotly


In [2]:

from google.colab import files

uploaded = files.upload()


Saving world_happiness_2023.csv to world_happiness_2023.csv


In [3]:

import pandas as pd

df = pd.read_csv("world_happiness_2023.csv")

df.columns = [
    "Country", "Region", "Score", "GDP",
    "Social_Support", "Life_Expectancy",
    "Freedom", "Generosity", "Corruption"
]

df.head()


,Country,Region,Score,GDP,Social_Support,Life_Expectancy,Freedom,Generosity,Corruption
0,Finland,Western Europe,7.804,10.775,0.954,71.9,0.949,0.142,0.179
1,Denmark,Western Europe,7.586,10.933,0.954,72.7,0.931,0.168,0.234
2,Iceland,Western Europe,7.525,10.878,0.983,72.5,0.961,0.260,0.150
3,Israel,Middle East and North Africa,7.473,10.527,0.916,72.4,0.903,0.149,0.826
4,Netherlands,Western Europe,7.464,11.015,0.939,72.4,0.879,0.240,0.296


In [4]:

global_average = df["Score"].mean()

print("Global average happiness:", round(global_average, 2))

df["Difference"] = df["Score"] - global_average

df[["Country", "Score", "Difference"]].head()


Global average happiness: 5.81


,Country,Score,Difference
0,Finland,7.804,1.993286
1,Denmark,7.586,1.775286
2,Iceland,7.525,1.714286
3,Israel,7.473,1.662286
4,Netherlands,7.464,1.653286


In [5]:

import plotly.express as px

comparison = df.sort_values("Difference")

max_difference = df["Difference"].abs().max()

fig = px.bar(
    comparison,
    x="Difference",
    y="Country",
    orientation="h",
    color="Difference",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    range_color=[-max_difference, max_difference],
    title="Happiness Compared to the Global Average",
    height=2200
)

fig.add_vline(
    x=0,
    line_dash="dash",
    line_color="gray",
    annotation_text=f"Global average: {global_average:.2f}",
    annotation_position="top"
)

fig.show()


In [6]:

%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px
from pathlib import Path

# =====================================
# PAGE CONFIGURATION
# =====================================

st.set_page_config(
    page_title="World Happiness Report",
    page_icon="🌍",
    layout="wide"
)

st.title("🌍 World Happiness Dashboard")

st.caption("Source: World Happiness Report 2023 | Kaggle")

# =====================================
# LOAD DATA
# =====================================

DATA = Path(__file__).parent / "world_happiness_2023.csv"

df = pd.read_csv(DATA)

df.columns = [
    "Country", "Region", "Score", "GDP",
    "Social_Support", "Life_Expectancy",
    "Freedom", "Generosity", "Corruption"
]

# =====================================
# SIDEBAR FILTERS
# =====================================

with st.sidebar:

    st.header("Filters")

    regions = ["All Regions"] + sorted(
        df["Region"].unique().tolist()
    )

    selected_region = st.selectbox(
        "Region", regions
    )

    top_n = st.slider(
        "Show top N countries", 5, 25, 15
    )

# Filter data by selected region

filtered = (
    df if selected_region == "All Regions"
    else df[df["Region"] == selected_region]
)

top = filtered.nlargest(
    top_n, "Score"
).sort_values("Score")

# =====================================
# KPI METRICS
# =====================================

col1, col2, col3 = st.columns(3)

col1.metric(
    "Countries",
    len(filtered)
)

col2.metric(
    "Avg Score",
    f"{filtered['Score'].mean():.2f}",
    f"{filtered['Score'].mean() - df['Score'].mean():+.2f} vs global"
)

col3.metric(
    "Happiest",
    filtered.nlargest(1, "Score")["Country"].values[0]
)

st.divider()

# =====================================
# CHART 1: PROFESSOR DINA'S CHART
# =====================================

fig1 = px.bar(
    top,
    x="Score",
    y="Country",
    orientation="h",
    labels={
        "Score": "Happiness Score (0–10)",
        "Country": ""
    },
    template="plotly_white",
    title=f"Happiest {len(top)} Countries - {selected_region}",
    height=600
)

fig1.update_layout(
    xaxis=dict(range=[0, 8.0])
)

st.plotly_chart(fig1, width="stretch")

st.divider()

# =====================================
# CHART 2: MY CLASS EXERCISE
# =====================================

st.header("Happiness Compared to the Global Average")

st.write(
    "This chart compares each country's happiness "
    "score with the global average. Red represents "
    "below-average happiness and green represents "
    "above-average happiness."
)

# Calculate global average

global_average = df["Score"].mean()

# Calculate happiness difference

comparison = filtered.copy()

comparison["Difference"] = (
    comparison["Score"] - global_average
)

comparison = comparison.sort_values("Difference")

# Set a symmetric colour range

max_difference = (
    df["Score"] - global_average
).abs().max()

# Create diverging bar chart

fig2 = px.bar(
    comparison,
    x="Difference",
    y="Country",
    orientation="h",
    color="Difference",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    range_color=[-max_difference, max_difference],
    title=f"Happiness Gap - {selected_region}",
    labels={
        "Difference": "Difference from Global Average",
        "Country": ""
    },
    template="plotly_white",
    height=max(500, len(comparison) * 22)
)

# Annotate the midpoint

fig2.add_vline(
    x=0,
    line_dash="dash",
    line_color="gray",
    annotation_text=f"Global average: {global_average:.2f}",
    annotation_position="top"
)

# Display second chart

st.plotly_chart(fig2, width="stretch")

st.divider()

st.caption("Built with Streamlit + Plotly")


Writing app.py


In [7]:

%%writefile requirements.txt
streamlit
pandas
plotly


Writing requirements.txt


In [8]:

import os

for name in [
    "app.py",
    "requirements.txt",
    "world_happiness_2023.csv"
]:
    print(name, ":", os.path.exists(name))


app.py : True
requirements.txt : True
world_happiness_2023.csv : True


In [12]:

from google.colab import files

files.download("app.py")
files.download("requirements.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>